In [1]:
import sys
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false" 

import pandas as pd

sys.path.append(os.path.abspath(os.path.join('..')))

sys.modules.pop("functionality.models", None)
sys.modules.pop("functionality.data_preparation", None)
from functionality.data_preparation import EmbDataset, train_model
from functionality.models import ChemBertaBinaryClassifierLightning

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger
from transformers import AutoTokenizer, AutoModel

import torch
import pickle
from tqdm import tqdm
import gc

/home/user/Desktop/Pharm/Medicine/BELKA/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
binds_0 = pd.read_parquet("../intermediates/downsampled_0_50_mln")
binds_1 = pd.read_parquet("../intermediates/1_class")
final_data = pd.concat([binds_0, binds_1], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
del binds_1
del binds_0

In [3]:
def compute_and_save_embeddings(model_name, data, save_path, batch_size):
    """
    Compute embeddings for a list of SMILES strings in batches and save them

    Args:
        model_name (str): Pretrained ChemBERTa model name
        data (pd.Series):data with SMILES strings
        save_path (str): Path to save computed embeddings
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    with open(save_path, "wb") as f:
        for i in tqdm(range(0, len(data), batch_size), desc="Computing Embeddings"):
            batch_smiles = data[i : i + batch_size]
            batch_smiles = batch_smiles.tolist()
            tokens = tokenizer(batch_smiles, padding=True, truncation=True, max_length=512, return_tensors="pt")

            with torch.no_grad():
                outputs = model(**tokens)

            batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu()
            pickle.dump(batch_embeddings, f)

            del batch_smiles, tokens, outputs, batch_embeddings
            gc.collect()

    print(f"Embeddings saved to {save_path}")
    return save_path

In [17]:
def train_model(final_data, model_name="seyonec/PubChem10M_SMILES_BPE_450k", batch_size=10000, save_dir="../intermediates"):
    protein_names = final_data.protein_name.unique()

    for protein_name in protein_names:
        print(f"Training model for protein: {protein_name}")

        protein_data = final_data[final_data.protein_name == protein_name]
        train_data, val_data = train_test_split(protein_data, test_size=0.1, random_state=42)

        train_embeddings_path = os.path.join(save_dir, f"{protein_name}_train_embeddings.pkl")
        val_embeddings_path = os.path.join(save_dir, f"{protein_name}_val_embeddings.pkl")

        train_embeddings_path = compute_and_save_embeddings(model_name, train_data['molecule_smiles'], train_embeddings_path, batch_size)
        val_embeddings_path = compute_and_save_embeddings(model_name, val_data["molecule_smiles"], val_embeddings_path, batch_size)

        train_dataset = EmbDataset(train_data, train_embeddings_path)
        val_dataset = EmbDataset(val_data, val_embeddings_path)

        train_loader = DataLoader(train_dataset, batch_size=100000, shuffle=True, num_workers=4)
        val_loader = DataLoader(val_dataset, batch_size=100000, shuffle=False, num_workers=4)

        logger = CSVLogger("logs", name=model_name)
        early_stopping = EarlyStopping(monitor="val_loss", patience=3, mode="min")
        checkpoint_callback = ModelCheckpoint(
            dirpath="../checkpoints",
            filename=f"{protein_name}_{model_name}-{{epoch}}-{{val_loss:.4f}}",
            monitor="val_loss",
            save_top_k=1,
            mode="min",
            save_last=True,
            verbose=True,
        )

        trainer = pl.Trainer(
            max_epochs=20,
            accelerator="auto",
            devices=1,
            log_every_n_steps=2,
            callbacks=[early_stopping, checkpoint_callback],
            logger=logger,
        )


        chemberta_model = ChemBertaBinaryClassifierLightning()
        trainer.fit(chemberta_model, train_loader, val_loader)

        os.makedirs("../models", exist_ok=True)
        trainer.save_checkpoint(f"../models/{protein_name}_{model_name}.ckpt")

        print(f"Completed training for protein: {protein_name}")

        del train_data, val_data, train_dataset, val_dataset, train_loader, val_loader, chemberta_model
        gc.collect()

In [18]:
train_model(final_data)

Training model for protein: BRD4


Computing Embeddings:   0%|          | 0/342 [00:00<?, ?it/s]

: 